# Gold — dim_ticker (SCD Type 2)

`silver.ticker` → **`gold.dim_ticker`**.

One row per **version** of a ticker. All 118 trusts plus SPY, IVV and VOO — including the 17
with no usable prices and the 6 excluded, so the universe stays honest about what existed.

A new version opens when **`manager`, `management_group` or `status`** changes. Nothing else
versions: `aic_sector` never changes, and the coverage counts change every month, which would
version every trust monthly — noise dressed as history.

The manager file is a **snapshot with no dates**, so the first run makes everything version 1
and history accrues from the second run onward. Proving it works is the demo: change one
manager, re-run, and that ticker has two rows.

Expected on the first run: **121 rows, all current**.

In [0]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.dim_ticker (
  ticker_key            STRING  COMMENT 'MD5(ticker|effective_start_month). The key the facts carry',
  ticker                STRING  COMMENT 'Business key. Not unique here: one row per version',
  trust_name            STRING,
  entity_type           STRING  COMMENT 'Trust or Index, so trust-vs-index is a self-join',
  aic_sector            STRING,
  manager               STRING  COMMENT 'SCD2 driver. NoInfo or NotApplicable, never blank',
  management_group      STRING  COMMENT 'SCD2 driver',
  manager_structure     STRING  COMMENT 'sole or multi, from the count of named managers',
  status                STRING  COMMENT 'SCD2 driver. active or delisted: the survivorship column',
  currency              STRING,
  price_source          STRING  COMMENT 'yahoo, archive or none',
  data_status           STRING  COMMENT 'usable, stub, no-data or excluded',
  source_url            STRING  COMMENT 'Where the metadata came from. Cited, not assumed',
  first_month           INT,
  last_month            INT,
  months_available      INT,
  effective_start_month INT     COMMENT 'YYYYMM this version became true',
  effective_end_month   INT     COMMENT 'YYYYMM it stopped being true. Null while current',
  is_current            BOOLEAN
)
COMMENT 'Every trust and index, versioned on manager, management group and status';

In [0]:
CREATE OR REPLACE TEMP VIEW gold_stage_ticker AS
WITH clock AS (
  -- The last complete month. A change detected now takes effect from the month after it,
  -- so a new version never overlaps the one it replaces.
  SELECT MAX(month_key)                                                    AS latest_month,
         CAST(DATE_FORMAT(ADD_MONTHS(MAX(month_start), 1), 'yyyyMM') AS INT) AS next_month,
         CAST(DATE_FORMAT(MIN(month_start), 'yyyyMM') AS INT)              AS earliest_month
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
labelled AS (
  -- Absence is labelled, never left blank or null, and the two kinds are kept apart:
  -- an index has no manager because it is an index, which is not missing information.
  SELECT s.ticker, s.trust_name, s.entity_type,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.aic_sector), ''), 'NoInfo') END        AS aic_sector,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.manager), ''), 'NoInfo') END           AS manager,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.management_group), ''), 'NoInfo') END  AS management_group,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.manager_structure), ''), 'NoInfo') END AS manager_structure,
         s.status,
         CASE WHEN s.entity_type = 'Index' THEN s.currency
              ELSE COALESCE(NULLIF(TRIM(s.currency), ''), 'NoInfo') END          AS currency,
         s.price_source, s.data_status,
         CASE WHEN s.entity_type = 'Index' THEN 'NotApplicable'
              ELSE COALESCE(NULLIF(TRIM(s.source_url), ''), 'NoInfo') END        AS source_url,
         s.first_month, s.last_month, s.months_available
  FROM `index-vs-trust-pipeline`.silver.ticker s
),
dated AS (
  SELECT l.*,
         c.latest_month,
         -- A ticker we have never seen starts at its own first month: the snapshot is
         -- assumed to have held from the beginning, and the spec says so openly. Any later
         -- version starts in the month after the change was detected.
         CASE WHEN EXISTS (SELECT 1 FROM `index-vs-trust-pipeline`.gold.dim_ticker d
                           WHERE d.ticker = l.ticker)
              THEN c.next_month
              ELSE COALESCE(l.first_month, c.earliest_month)
         END AS effective_start_month
  FROM labelled l CROSS JOIN clock c
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(effective_start_month AS STRING))) AS ticker_key,
       ticker, trust_name, entity_type, aic_sector, manager, management_group,
       manager_structure, status, currency, price_source, data_status, source_url,
       first_month, last_month, months_available,
       effective_start_month,
       CAST(NULL AS INT) AS effective_end_month,
       true              AS is_current,
       latest_month
FROM dated;

## The SCD2, in two statements

**First close the old version, then open the new one.** Two plain statements rather than the
usual single MERGE with a null merge key — the clever version is one statement, this version
is one sentence.

The second statement inserts for a brand-new ticker **and** for one whose current version the
first statement just closed, because in both cases no current row is left to match.

In [0]:
-- 1. Close any version whose driver attributes have changed.
--    Null-safe (<=>) so a missing manager is never mistaken for a change.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_ticker AS t
USING gold_stage_ticker AS s
   ON t.ticker = s.ticker AND t.is_current
WHEN MATCHED AND NOT (t.manager          <=> s.manager
                  AND t.management_group <=> s.management_group
                  AND t.status           <=> s.status)
THEN UPDATE SET t.is_current          = false,
                t.effective_end_month = s.latest_month;

In [0]:
-- 2. Open a version for anything with no current row: new tickers, and the ones just closed.
MERGE INTO `index-vs-trust-pipeline`.gold.dim_ticker AS t
USING gold_stage_ticker AS s
   ON t.ticker = s.ticker AND t.is_current
WHEN NOT MATCHED THEN INSERT (
  ticker_key, ticker, trust_name, entity_type, aic_sector, manager, management_group,
  manager_structure, status, currency, price_source, data_status, source_url,
  first_month, last_month, months_available,
  effective_start_month, effective_end_month, is_current
) VALUES (
  s.ticker_key, s.ticker, s.trust_name, s.entity_type, s.aic_sector, s.manager,
  s.management_group, s.manager_structure, s.status, s.currency, s.price_source,
  s.data_status, s.source_url, s.first_month, s.last_month, s.months_available,
  s.effective_start_month, s.effective_end_month, s.is_current
);

## Verification

In [0]:
SELECT COUNT(*)                                                      AS rows_total,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END)                   AS current_rows,
       SUM(CASE WHEN effective_end_month IS NOT NULL THEN 1 ELSE 0 END) AS closed_rows,
       COUNT(*) - COUNT(DISTINCT ticker_key)                         AS duplicate_keys,
       COUNT(DISTINCT ticker)                                        AS distinct_tickers,
       SUM(CASE WHEN entity_type = 'Index'     THEN 1 ELSE 0 END)    AS index_rows,
       SUM(CASE WHEN status = 'delisted'       THEN 1 ELSE 0 END)    AS delisted,
       SUM(CASE WHEN data_status = 'excluded'  THEN 1 ELSE 0 END)    AS excluded
FROM `index-vs-trust-pipeline`.gold.dim_ticker;

Expect **121 / 121 / 0 / 0 / 121 / 3 / 19 / 6** on the first run.

`rows_total`, `current_rows` and `distinct_tickers` are all **121** only while every ticker is
on version 1. After the SCD2 demo, `rows_total` rises and `distinct_tickers` stays at 121 —
that gap *is* the history.

In [0]:
-- Nothing is blank, and the two kinds of absence stay apart.
SELECT SUM(CASE WHEN manager = 'NoInfo'                THEN 1 ELSE 0 END) AS manager_noinfo,
       SUM(CASE WHEN management_group = 'NoInfo'       THEN 1 ELSE 0 END) AS group_noinfo,
       SUM(CASE WHEN manager = 'NotApplicable'         THEN 1 ELSE 0 END) AS manager_na,
       SUM(CASE WHEN manager_structure = 'multi'       THEN 1 ELSE 0 END) AS multi,
       SUM(CASE WHEN manager_structure = 'sole'        THEN 1 ELSE 0 END) AS sole,
       COUNT(DISTINCT management_group)                                   AS distinct_groups,
       SUM(CASE WHEN manager IS NULL OR management_group IS NULL
                  OR manager_structure IS NULL THEN 1 ELSE 0 END)         AS any_nulls_left,
       SUM(CASE WHEN source_url LIKE 'http%'           THEN 1 ELSE 0 END) AS real_source_urls
FROM `index-vs-trust-pipeline`.gold.dim_ticker;

Expect **21 / 19 / 3 / 70 / 27 / 54 / 0 / 118**.

- `any_nulls_left` must be **0** — every row carries a readable label.
- `distinct_groups` is **54**: the 52 real management groups, plus `NoInfo` and
  `NotApplicable`. That is deliberate, so the dashboard can show how many trusts publish no
  group rather than hiding them in a null.
- `real_source_urls` is **118** — every trust cites where its metadata came from.

In [0]:
-- The survivorship evidence, and the star's whole point: status lives in the dimension, so
-- "with and without delisted trusts" is one fact table and two WHERE clauses.
SELECT status, data_status, COUNT(*) AS tickers
FROM `index-vs-trust-pipeline`.gold.dim_ticker
WHERE is_current
GROUP BY status, data_status
ORDER BY status, data_status;

Expect the delisted rows to total **19** — the 16 Yahoo erased, plus `BCPT` and `CSH` which
stopped being priced, plus `ADIG`.

This is the table to put on screen when asked *"how do you know those trusts existed?"*

In [0]:
-- Four rows that each stress a different part of the design.
SELECT ticker, entity_type, manager, management_group, manager_structure,
       status, data_status, price_source, months_available,
       effective_start_month, effective_end_month, is_current
FROM `index-vs-trust-pipeline`.gold.dim_ticker
WHERE ticker IN ('SMT', 'SPY', 'BCPT', 'ABR')
ORDER BY CASE ticker WHEN 'SMT' THEN 1 WHEN 'SPY' THEN 2 WHEN 'BCPT' THEN 3 ELSE 4 END;

Expect:

| ticker | why it is here |
|---|---|
| `SMT` | the ordinary case — Baillie Gifford, `multi`, active, 177 months |
| `SPY` | why `NotApplicable` exists: an index has no manager **because it is an index** |
| `BCPT` | delisted, recovered from the archive, no manager in the file → `NoInfo` |
| `ABR` | one of the 18 Yahoo erased — **no prices at all**, and still in the dimension |

`ABR` is the row that matters. A pipeline that only loaded what came back would not know it
ever existed, and that is the bias this project is about.